# HeritageAI: AI System for Automatic Identification and Classification of Cultural Patterns
### End-to-End Research Pipeline Notebook

**Author**: Senior AI/ML Engineer & Computer Vision Researcher  
**Foundation**: Research Paper and Patent Application  
**Domain**: Digital Heritage Preservation & Pattern Recognition for Indian Cultural Motifs  

---

## Notebook Overview
This Jupyter Notebook implements the complete end-to-end computer vision and deep learning pipeline proposed in the research paper for identifying three Indian cultural pattern categories:
1. **Warli Art Motifs** (Tribal geometric stick figures, tarpa dancers)
2. **Madhubani Paintings** (Double outlines, fish/floral motifs, Mithila line hatchings)
3. **Indian Textile Patterns** (Paisley/Kalka, Bandhani tie-dye grid dots, block prints)

### Included Modules:
- **Section 1**: Environment setup & reproducible random seeds
- **Section 2**: Data preparation & synthetic cultural pattern sample generation
- **Section 3**: Research image preprocessing (aspect ratio preservation, 224x224x3 RGB, [0,1] normalization, training augmentation)
- **Section 4**: Proposed HeritageCNN architecture definition (Stacked Conv2D, MaxPool, Dropout 0.45, Adam lr=0.0001)
- **Section 5**: Model training with validation monitoring & early stopping (patience 8)
- **Section 6**: Model evaluation (Accuracy, Macro Precision/Recall/F1, Confusion Matrix) & paper baseline comparison
- **Section 7**: Explainable AI (Grad-CAM spatial attention heatmap visualization)
- **Section 8**: Single-image inference pipeline & low-confidence warning demonstration

--- 
## Section 1: Environment & Dependency Setup
Import essential computer vision, deep learning, evaluation, and plotting libraries.

In [ ]:
import os
import time
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Set reproducible random seeds
def set_seeds(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[Setup] Random seeds set to {seed}")

set_seeds()
print(f"[TensorFlow Version] {tf.__version__}")

--- 
## Section 2: Data Preparation & Sample Dataset Generation
To ensure this notebook is fully runnable without downloading external data, we generate legitimate sample geometric artwork images for Warli, Madhubani, and Indian Textile motifs.

In [ ]:
def create_warli_sample(size=(224, 224)) -> np.ndarray:
    """Generates Warli art style pattern: earthy background with white geometric stick motifs."""
    bg_color = (165, 60, 35)
    img = np.full((size[1], size[0], 3), bg_color, dtype=np.uint8)
    white = (245, 245, 240)
    center = (size[0] // 2, size[1] // 2)
    cv2.circle(img, center, 25, white, 2)
    for angle in np.linspace(0, 2*np.pi, 8, endpoint=False):
        radius = 55
        fx = int(center[0] + radius * np.cos(angle))
        fy = int(center[1] + radius * np.sin(angle))
        cv2.circle(img, (fx, fy - 12), 4, white, -1)
        pts1 = np.array([[fx, fy - 8], [fx - 6, fy], [fx + 6, fy]], np.int32)
        cv2.fillPoly(img, [pts1], white)
        pts2 = np.array([[fx, fy + 8], [fx - 6, fy], [fx + 6, fy]], np.int32)
        cv2.fillPoly(img, [pts2], white)
        cv2.line(img, (fx, fy + 8), (fx - 5, fy + 18), white, 2)
        cv2.line(img, (fx, fy + 8), (fx + 5, fy + 18), white, 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def create_madhubani_sample(size=(224, 224)) -> np.ndarray:
    """Generates Madhubani art style pattern: double outlines, fish motifs, vibrant colors."""
    bg_color = (250, 243, 224)
    img = np.full((size[1], size[0], 3), bg_color, dtype=np.uint8)
    black = (20, 20, 20)
    bright_red = (200, 30, 40)
    bright_yellow = (240, 200, 40)
    center = (size[0] // 2, size[1] // 2)
    cv2.rectangle(img, (10, 10), (size[0]-10, size[1]-10), black, 2)
    cv2.rectangle(img, (14, 14), (size[0]-14, size[1]-14), black, 1)
    cv2.ellipse(img, center, (60, 30), 0, 0, 360, bright_yellow, -1)
    cv2.ellipse(img, center, (60, 30), 0, 0, 360, black, 2)
    cv2.ellipse(img, center, (56, 27), 0, 0, 360, black, 1)
    tail_pts = np.array([[center[0]-55, center[1]], [center[0]-85, center[1]-25], [center[0]-85, center[1]+25]], np.int32)
    cv2.fillPoly(img, [tail_pts], bright_red)
    cv2.polylines(img, [tail_pts], True, black, 2)
    for x_off in range(-30, 30, 8):
        cv2.line(img, (center[0]+x_off, center[1]-15), (center[0]+x_off, center[1]+15), black, 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def create_textile_sample(size=(224, 224)) -> np.ndarray:
    """Generates Indian Textile pattern: Paisley/Kalka motifs and Bandhani dots."""
    bg_color = (25, 20, 80)
    img = np.full((size[1], size[0], 3), bg_color, dtype=np.uint8)
    gold = (212, 175, 55)
    cream = (245, 240, 230)
    for y in range(20, size[1]-15, 25):
        for x in range(20, size[0]-15, 25):
            cv2.circle(img, (x, y), 2, cream, -1)
    center = (size[0] // 2, size[1] // 2)
    cv2.ellipse(img, center, (40, 60), 20, 0, 360, gold, 3)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Create sample dataset directories
DATASET_DIR = "nb_datasets/cultural_patterns"
CLASS_NAMES = ["warli", "madhubani", "textile"]
CLASS_DISPLAY = {"warli": "Warli Art", "madhubani": "Madhubani Painting", "textile": "Indian Textile"}

generators = {"warli": create_warli_sample, "madhubani": create_madhubani_sample, "textile": create_textile_sample}
splits = {"train": 60, "validation": 15, "test": 15}

for split, count in splits.items():
    for cls in CLASS_NAMES:
        path = os.path.join(DATASET_DIR, split, cls)
        os.makedirs(path, exist_ok=True)
        gen_func = generators[cls]
        for i in range(count):
            img = gen_func()
            angle = np.random.uniform(-10, 10)
            M = cv2.getRotationMatrix2D((112, 112), angle, 1.0)
            var_img = cv2.warpAffine(img, M, (224, 224), borderMode=cv2.BORDER_REFLECT)
            Image.fromarray(var_img).save(os.path.join(path, f"{cls}_{i+1:03d}.png"))

print(f"[Dataset] Successfully created sample dataset in '{DATASET_DIR}'")

In [ ]:
# Plot representative samples from each class
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for idx, cls in enumerate(CLASS_NAMES):
    sample_path = os.path.join(DATASET_DIR, "train", cls, f"{cls}_001.png")
    img = Image.open(sample_path)
    axes[idx].imshow(img)
    axes[idx].set_title(f"Class: {CLASS_DISPLAY[cls]}", fontsize=12, fontweight="bold")
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

--- 
## Section 3: Image Preprocessing Pipeline
Implements the exact preprocessing parameters defined in the research paper:
1. Aspect-ratio preserving resize with zero padding (letterboxing) to 224x224x3 RGB.
2. Scaling pixel values from [0, 255] to floating-point [0.0, 1.0].
3. Augmentation pipeline for training (rotation up to ±15°, shearing, Gaussian blur).

In [ ]:
def preprocess_image(image_path, target_size=(224, 224)) -> np.ndarray:
    """Preprocesses an image for CNN inference following research paper specification."""
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    if img.mode != "RGB":
        img = img.convert("RGB")
        
    np_img = np.array(img)
    h, w = np_img.shape[:2]
    target_w, target_h = target_size
    scale = min(target_w / w, target_h / h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    
    resized = cv2.resize(np_img, (nw, nh), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    top = (target_h - nh) // 2
    left = (target_w - nw) // 2
    canvas[top:top+nh, left:left+nw] = resized
    
    # Scale [0, 255] -> [0.0, 1.0]
    normalized = canvas.astype(np.float32) / 255.0
    return normalized

def apply_training_augmentation(image: np.ndarray) -> np.ndarray:
    """Applies paper-specified data augmentation: ±15° rotation, shearing, Gaussian blur."""
    h, w = image.shape[:2]
    img_uint8 = (image * 255.0).astype(np.uint8)
    
    # Rotation ±15 deg
    angle = np.random.uniform(-15.0, 15.0)
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    aug = cv2.warpAffine(img_uint8, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
    
    # Shearing
    if np.random.rand() > 0.5:
        shear_factor = np.random.uniform(-0.1, 0.1)
        shear_mat = np.float32([[1, shear_factor, 0], [0, 1, 0]])
        aug = cv2.warpAffine(aug, shear_mat, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
        
    # Controlled Gaussian blur
    if np.random.rand() > 0.6:
        aug = cv2.GaussianBlur(aug, (3, 3), 0)
        
    return aug.astype(np.float32) / 255.0

# Preprocess and augment sample
sample_p = os.path.join(DATASET_DIR, "train/warli/warli_001.png")
norm_img = preprocess_image(sample_p)
aug_img = apply_training_augmentation(norm_img)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(norm_img)
ax[0].set_title("Preprocessed Normalized (224x224x3)")
ax[0].axis("off")
ax[1].imshow(aug_img)
ax[1].set_title("Training Augmented (±15° Rot + Shear)")
ax[1].axis("off")
plt.show()

--- 
## Section 4: Proposed HeritageCNN Architecture
Defines the TensorFlow/Keras CNN architecture strictly matching the research paper parameters.

In [ ]:
def build_heritage_cnn(input_shape=(224, 224, 3), num_classes=3, learning_rate=0.0001, dropout_rate=0.45):
    """
    Builds proposed HeritageCNN model:
    - Input: 224x224x3 RGB
    - Stacked Conv2D blocks with ReLU activations
    - 2x2 Max-pooling (stride 2)
    - Target activation layer 'target_conv_layer' for Grad-CAM
    - Flatten + Dense(256, relu)
    - Dropout(0.45)
    - Output Dense(3, Softmax)
    """
    model = models.Sequential(name="HeritageCNN")
    model.add(layers.Input(shape=input_shape))
    
    # Conv Block 1
    model.add(layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv2d_b1_1"))
    model.add(layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv2d_b1_2"))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_1"))
    
    # Conv Block 2
    model.add(layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2d_b2_1"))
    model.add(layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2d_b2_2"))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_2"))
    
    # Conv Block 3 (Target layer for Grad-CAM)
    model.add(layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv2d_b3_1"))
    model.add(layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="target_conv_layer"))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2), name="maxpool_3"))
    
    # Classifier
    model.add(layers.Flatten(name="flatten"))
    model.add(layers.Dense(256, activation="relu", name="dense_fc1"))
    model.add(layers.Dropout(rate=dropout_rate, name="dropout"))
    model.add(layers.Dense(num_classes, activation="softmax", name="output_softmax"))
    
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss=losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    return model

model = build_heritage_cnn()
model.summary()

--- 
## Section 5: Model Training & Validation Monitoring
Trains the CNN using custom DataSequence generator, early stopping (patience 8), and checkpointing.

In [ ]:
class HeritageDataSequence(tf.keras.utils.Sequence):
    def __init__(self, data_dir, batch_size=16, is_training=True):
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.is_training = is_training
        self.image_paths = []
        self.labels = []
        for idx, cls in enumerate(CLASS_NAMES):
            cls_p = os.path.join(data_dir, cls)
            for f in os.listdir(cls_p):
                if f.endswith('.png'):
                    self.image_paths.append(os.path.join(cls_p, f))
                    self.labels.append(idx)
        self.indexes = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = [preprocess_image(self.image_paths[i]) for i in batch_idx]
        if self.is_training:
            batch_x = [apply_training_augmentation(x) for x in batch_x]
        batch_y = [self.labels[i] for i in batch_idx]
        return np.array(batch_x, dtype=np.float32), np.array(batch_y, dtype=np.int32)

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.indexes)

train_seq = HeritageDataSequence(os.path.join(DATASET_DIR, "train"), batch_size=16, is_training=True)
val_seq = HeritageDataSequence(os.path.join(DATASET_DIR, "validation"), batch_size=16, is_training=False)

# Callbacks
cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("heritage_cnn_notebook.h5", monitor="val_accuracy", save_best_only=True, verbose=1)
]

# Fit Model
history = model.fit(train_seq, validation_data=val_seq, epochs=25, callbacks=cb)

In [ ]:
# Plot Training Loss and Accuracy Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='#B85C38', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='#171C3A', linewidth=2)
plt.title('HeritageCNN Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#B85C38', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='#171C3A', linewidth=2)
plt.title('HeritageCNN Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

--- 
## Section 6: Held-Out Test Evaluation & Paper Baseline Comparison
Evaluates the model on the test dataset split and displays classification metrics and confusion matrix.

In [ ]:
test_seq = HeritageDataSequence(os.path.join(DATASET_DIR, "test"), batch_size=1, is_training=False)

y_true = []
y_pred = []
latencies = []

for i in range(len(test_seq)):
    x, y = test_seq[i]
    t0 = time.time()
    preds = model.predict(x, verbose=0)
    t1 = time.time()
    y_true.append(y[0])
    y_pred.append(np.argmax(preds[0]))
    latencies.append(t1 - t0)

acc = accuracy_score(y_true, y_pred)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
cm = confusion_matrix(y_true, y_pred)

print(f"=== NEWLY EVALUATED MODEL METRICS ===")
print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"Macro Precision:  {macro_p:.4f}")
print(f"Macro Recall:     {macro_r:.4f}")
print(f"Macro F1-Score:   {macro_f1:.4f}")
print(f"Average Latency:  {np.mean(latencies):.4f} seconds/image")

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[CLASS_DISPLAY[c] for c in CLASS_NAMES], yticklabels=[CLASS_DISPLAY[c] for c in CLASS_NAMES])
plt.title("Confusion Matrix (Test Split)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()

In [ ]:
# Historical Research Paper Baseline Benchmark Comparison Table
import pandas as pd

paper_data = {
    "Architecture / Model": ["Proposed HeritageCNN (Paper Result)", "AlexNet Baseline", "ResNet-50 Baseline", "Newly Evaluated Model"],
    "Mean Accuracy": ["96.2%", "79.5%", "86.8%", f"{acc*100:.1f}%"],
    "Macro Precision": ["0.958", "0.781", "0.865", f"{macro_p:.3f}"],
    "Macro Recall": ["0.946", "0.764", "0.851", f"{macro_r:.3f}"],
    "Macro F1-Score": ["0.952", "0.772", "0.858", f"{macro_f1:.3f}"],
    "Inference Latency": ["0.22s", "0.12s", "0.45s", f"{np.mean(latencies):.2f}s"],
    "Metric Source": ["Published Paper Benchmark", "Published Paper Benchmark", "Published Paper Benchmark", "Newly Evaluated Run"]
}

df_comparison = pd.DataFrame(paper_data)
display(df_comparison)

--- 
## Section 7: Explainable AI — Grad-CAM Heatmap Visualization
Generates spatial Grad-CAM attention heatmaps from the target convolutional layer (`target_conv_layer`) to highlight visual features influencing the network's prediction.

In [ ]:
def generate_gradcam(model, img_array, layer_name="target_conv_layer"):
    grad_model = tf.keras.models.Model(inputs=[model.inputs], outputs=[model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]
        
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Test Grad-CAM on sample image
sample_test = os.path.join(DATASET_DIR, "test/madhubani/madhubani_001.png")
test_tensor = np.expand_dims(preprocess_image(sample_test), axis=0)
heatmap = generate_gradcam(model, test_tensor)

heatmap_resized = cv2.resize(heatmap, (224, 224))
heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
orig_img = np.uint8(255 * test_tensor[0])
superimposed = cv2.addWeighted(orig_img, 0.6, heatmap_colored, 0.4, 0)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(test_tensor[0])
ax[0].set_title("Original Preprocessed Image")
ax[0].axis("off")
ax[1].imshow(heatmap_resized, cmap="jet")
ax[1].set_title("Grad-CAM Attention Heatmap")
ax[1].axis("off")
ax[2].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
ax[2].set_title("Superimposed Overlay")
ax[2].axis("off")
plt.tight_layout()
plt.show()

--- 
## Section 8: Single Image Inference & Low-Confidence Warning
Defines the inference function with a configurable low-confidence warning threshold (0.60).

In [ ]:
def classify_cultural_pattern(image_path, model, confidence_threshold=0.60):
    """Performs full inference on a single image and checks low-confidence warning threshold."""
    tensor = np.expand_dims(preprocess_image(image_path), axis=0)
    t0 = time.time()
    preds = model.predict(tensor, verbose=0)[0]
    latency = time.time() - t0
    
    top_idx = int(np.argmax(preds))
    top_class = CLASS_NAMES[top_idx]
    top_display = CLASS_DISPLAY[top_class]
    confidence = float(preds[top_idx])
    low_conf_warning = confidence < confidence_threshold
    
    probs_dict = {CLASS_DISPLAY[c]: float(preds[i]) for i, c in enumerate(CLASS_NAMES)}
    
    print(f"=== CLASSIFICATION RESULT ===")
    print(f"Predicted Class:      {top_display}")
    print(f"Confidence Score:     {confidence*100:.2f}%")
    print(f"Inference Latency:    {latency:.4f}s")
    print(f"Low-Confidence Flag:  {'⚠️ WARNING: Human Expert Review Needed' if low_conf_warning else '✅ High Confidence'}")
    print(f"Full Softmax Output:  {probs_dict}")
    return top_class, confidence, probs_dict

# Run classification test
sample_file = os.path.join(DATASET_DIR, "test/textile/textile_002.png")
classify_cultural_pattern(sample_file, model)